# Big Data Graph Analytics with GraphFrames  
## Building and Understanding the Panama Papers Network

In this class we build a graph from the Panama Papers / Offshore Leaks data using **PySpark** and **GraphFrames**.

The Panama Papers were a major leak of documents from the Panamanian law firm Mossack Fonseca. The released data can be represented as a network: people, companies, intermediaries, and addresses are stored as nodes, and the relationship file connects them.

The main node types we use are:

| Node type | Meaning in this dataset |
|---|---|
| `entity` | Offshore company, trust, foundation, or similar legal structure |
| `officer` | Person or organisation linked to an entity |
| `intermediary` | Service provider or agent involved in creating/managing entities |
| `address` | Registered or contact address |

The goal is to learn how graph analytics can reveal hubs, repeated structures, and interesting subgraphs in a large connected dataset.


## Learning objectives

By the end of this class, we will be able to:

1. Explain vertices, edges, directed graphs, degree, in-degree, and out-degree.
2. Build a GraphFrame from real CSV files.
3. Interpret degree results by node type.
4. Understand long-tail networks.
5. Use `filterEdges` and `filterVertices` to create useful subgraphs.
6. Use country attributes to create focused summaries.
7. Use motif finding to search for repeated graph patterns.
8. Understand connected components using a small example.


# 1. Environment setup for Lightning.ai

Run the following cells before using GraphFrames.

These setup cells are designed for the Lightning.ai environment.


In [ ]:
# Checking the installed Java version

!java -version

In [ ]:
!pip install "pyspark==3.5.0"

In [ ]:
# Install Java 17

!sudo apt-get update

!sudo apt-get install -y openjdk-17-jdk-headless

!java -version

In [ ]:
%pip install graphframes-py==0.10.0

In [ ]:
# Set JAVA_HOME to Java 17

import os

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("GraphFramesWithSpark4") \
    .config("spark.jars.packages", "io.graphframes:graphframes-spark3_2.12:0.10.0") \
    .getOrCreate()

print(f"spark version: {spark.version}")

print("spark session created with graphframes package specified!")

# import the package we just installed
from graphframes import *

# import data types - All data types of Spark SQL are located in the package of pyspark.sql.types
from pyspark.sql.types import *

# Row can be used to create a row object by using named arguments
from pyspark.sql import Row

from pyspark.sql.functions import col

sc = spark.sparkContext

In [ ]:
# Additional imports used in this notebook

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# GraphFrames connected components requires a checkpoint directory
sc.setCheckpointDir("/tmp/graphframes-checkpoints")

spark.conf.set("spark.sql.shuffle.partitions", "16")

print("Ready.")

# 2. Graph basics using a tiny example

Before working with the real Panama Papers data, we will create a very small graph.

Imagine the following situation:

- Alice is a director of Company A.
- Bob is a shareholder of Company A.
- Company A has a registered address.
- Intermediary X is an intermediary of Company A.
- Company A is connected to Company B through a shared officer.

This toy example helps us understand the core GraphFrames requirements:

- A **vertices** DataFrame must contain a column called `id`.
- An **edges** DataFrame must contain columns called `src` and `dst`.


In [ ]:
toy_vertices = spark.createDataFrame([
    ("o1", "Alice", "officer"),
    ("o2", "Bob", "officer"),
    ("e1", "Company A", "entity"),
    ("e2", "Company B", "entity"),
    ("i1", "Intermediary X", "intermediary"),
    ("a1", "Address Z", "address"),
], ["id", "name", "node_type"])

toy_edges = spark.createDataFrame([
    ("o1", "e1", "director_of"),
    ("o2", "e1", "shareholder_of"),
    ("i1", "e1", "intermediary_of"),
    ("e1", "a1", "registered_address"),
    ("o1", "e2", "director_of"),
], ["src", "dst", "rel_type"])

toy_g = GraphFrame(toy_vertices, toy_edges)

print("Toy vertices")
toy_g.vertices.show(truncate=False)

print("Toy edges")
toy_g.edges.show(truncate=False)

## 2.1 Degree, in-degree, and out-degree

The **degree** of a node is the number of edges connected to it.

For a directed graph:

- **In-degree** = number of incoming edges.
- **Out-degree** = number of outgoing edges.

In this example:

- A company with high in-degree may have many officers, shareholders, or intermediaries pointing to it.
- An officer with high out-degree may be connected to many companies.


In [ ]:
print("Total degree")
toy_g.degrees.orderBy(F.desc("degree")).show()

print("In-degree")
toy_g.inDegrees.orderBy(F.desc("inDegree")).show()

print("Out-degree")
toy_g.outDegrees.orderBy(F.desc("outDegree")).show()

## 2.2 Toy graph interpretation


1. Which node is the most connected?
2. Is the most connected node necessarily the most suspicious?
3. What changes if we only keep `director_of` relationships?


In [ ]:
toy_director_edges = toy_g.edges.filter(F.col("rel_type") == "director_of")
toy_director_g = GraphFrame(toy_g.vertices, toy_director_edges)

toy_director_g.edges.show()
toy_director_g.degrees.orderBy(F.desc("degree")).show()

# 3. Load the Panama Papers CSV files

Update `DATA_DIR` if your CSV files are stored in a different folder.

Expected files:

```text
nodes_addresses.csv
nodes_entities.csv
nodes_intermediaries.csv
nodes_officers.csv
relationships.csv
```

The code below reads the CSV files using Spark.


In [ ]:
# Change this path if your files are stored elsewhere
DATA_DIR = "./data"

addresses_path = f"{DATA_DIR}/nodes_addresses.csv"
entities_path = f"{DATA_DIR}/nodes_entities.csv"
intermediaries_path = f"{DATA_DIR}/nodes_intermediaries.csv"
officers_path = f"{DATA_DIR}/nodes_officers.csv"
relationships_path = f"{DATA_DIR}/relationships.csv"

addresses_raw = spark.read.csv(addresses_path, header=True, inferSchema=True, multiLine=True, escape='"')
entities_raw = spark.read.csv(entities_path, header=True, inferSchema=True, multiLine=True, escape='"')
intermediaries_raw = spark.read.csv(intermediaries_path, header=True, inferSchema=True, multiLine=True, escape='"')
officers_raw = spark.read.csv(officers_path, header=True, inferSchema=True, multiLine=True, escape='"')
relationships_raw = spark.read.csv(relationships_path, header=True, inferSchema=True, multiLine=True, escape='"')

print("Loaded files.")
print("Addresses:", addresses_raw.count())
print("Entities:", entities_raw.count())
print("Intermediaries:", intermediaries_raw.count())
print("Officers:", officers_raw.count())
print("Relationships:", relationships_raw.count())

## 3.1 Inspect the raw data

Spark DataFrames can be very wide. We will inspect the schemas and a few rows.


In [ ]:
addresses_raw.printSchema()
entities_raw.printSchema()
intermediaries_raw.printSchema()
officers_raw.printSchema()
relationships_raw.printSchema()

In [ ]:
addresses_raw.show(3, truncate=80)
entities_raw.show(3, truncate=80)
intermediaries_raw.show(3, truncate=80)
officers_raw.show(3, truncate=80)
relationships_raw.show(3, truncate=80)

# 4. Build the vertices DataFrame

GraphFrames requires a vertices DataFrame with a column called `id`.

The Panama Papers-style node files use `node_id`, so we will rename it to `id`.

Because this is a heterogeneous graph, we will also add a column called `node_type`.


In [ ]:
def normalize_string_column(df, column_name):
    """Trim whitespace in a string column if the column exists."""
    if column_name in df.columns:
        return df.withColumn(column_name, F.trim(F.col(column_name).cast("string")))
    return df

def prepare_vertices(df, node_type, display_expr):
    """
    Convert a raw node DataFrame into a common vertex format.

    Required output columns:
    - id
    - name
    - node_type
    - countries
    - country_codes
    - sourceID
    - valid_until
    """
    out = (
        df
        .withColumn("id", F.col("node_id").cast("string"))
        .withColumn("node_type", F.lit(node_type))
        .withColumn("name", display_expr)
    )

    # Add optional columns if missing
    for c in ["countries", "country_codes", "sourceID", "valid_until", "jurisdiction", "jurisdiction_description", "status", "address"]:
        if c not in out.columns:
            out = out.withColumn(c, F.lit(None).cast("string"))
        else:
            out = out.withColumn(c, F.col(c).cast("string"))

    return out.select(
        "id",
        "name",
        "node_type",
        "countries",
        "country_codes",
        "sourceID",
        "valid_until",
        "jurisdiction",
        "jurisdiction_description",
        "status",
        "address"
    )

addresses_v = prepare_vertices(
    addresses_raw,
    "address",
    F.coalesce(
        F.col("name").cast("string"),
        F.col("address").cast("string"),
        F.concat(F.lit("Address node "), F.col("node_id").cast("string"))
    )
)

entities_v = prepare_vertices(
    entities_raw,
    "entity",
    F.coalesce(
        F.col("name").cast("string"),
        F.col("original_name").cast("string"),
        F.concat(F.lit("Entity node "), F.col("node_id").cast("string"))
    )
)

intermediaries_v = prepare_vertices(
    intermediaries_raw,
    "intermediary",
    F.coalesce(
        F.col("name").cast("string"),
        F.concat(F.lit("Intermediary node "), F.col("node_id").cast("string"))
    )
)

officers_v = prepare_vertices(
    officers_raw,
    "officer",
    F.coalesce(
        F.col("name").cast("string"),
        F.concat(F.lit("Officer node "), F.col("node_id").cast("string"))
    )
)

vertices = (
    addresses_v
    .unionByName(entities_v)
    .unionByName(intermediaries_v)
    .unionByName(officers_v)
    .dropDuplicates(["id"])
)

vertices.cache()

print("Total vertices:", vertices.count())
vertices.groupBy("node_type").count().orderBy("node_type").show()

# 5. Build the edges DataFrame

GraphFrames requires an edges DataFrame with columns:

- `src`: source node id
- `dst`: destination node id

The relationships file uses:

- `node_id_start`
- `node_id_end`

So we rename those columns.


In [ ]:
edges = (
    relationships_raw
    .withColumn("src", F.col("node_id_start").cast("string"))
    .withColumn("dst", F.col("node_id_end").cast("string"))
    .withColumn("rel_type", F.col("rel_type").cast("string"))
    .withColumn("link", F.col("link").cast("string"))
    .withColumn("sourceID", F.col("sourceID").cast("string"))
    .select("src", "dst", "rel_type", "link", "sourceID")
    .dropna(subset=["src", "dst"])
)

edges.cache()

print("Total edges:", edges.count())
edges.show(5, truncate=False)

## 5.1 Optional: focus only on Panama Papers records

The Offshore Leaks-style files may include multiple sources, for example Panama Papers, Bahamas Leaks, Paradise Papers, and others.

For this class, we usually want to focus on the Panama Papers subset.

Set `USE_PANAMA_PAPERS_ONLY = False` if you want to analyze the full dataset.


In [ ]:
USE_PANAMA_PAPERS_ONLY = True

if USE_PANAMA_PAPERS_ONLY:
    vertices_work = vertices.filter(F.col("sourceID") == "Panama Papers")
    edges_work = edges.filter(F.col("sourceID") == "Panama Papers")
else:
    vertices_work = vertices
    edges_work = edges

# Keep only edges where both endpoints exist in the selected vertices
valid_ids = vertices_work.select(F.col("id").alias("valid_id")).distinct()

edges_work = (
    edges_work
    .join(valid_ids, edges_work.src == valid_ids.valid_id, "left_semi")
    .join(valid_ids, edges_work.dst == valid_ids.valid_id, "left_semi")
)

vertices_work.cache()
edges_work.cache()

print("Working vertices:", vertices_work.count())
print("Working edges:", edges_work.count())

vertices_work.groupBy("node_type").count().orderBy("node_type").show()
edges_work.groupBy("rel_type").count().orderBy(F.desc("count")).show(20, truncate=False)

# 6. Create the Panama Papers GraphFrame


In [ ]:
g = GraphFrame(vertices_work, edges_work)

print("GraphFrame created.")
print("Vertices:", g.vertices.count())
print("Edges:", g.edges.count())

# 7. Descriptive graph analysis

Before running graph algorithms, we first inspect the basic structure of the graph.

The top-level counts help us check that the graph was built correctly. The real analysis starts when we look inside each node type: the biggest intermediaries, the most reused addresses, the most connected officers, and the countries associated with those nodes.


In [ ]:
print("Node counts by type")
g.vertices.groupBy("node_type").count().orderBy(F.desc("count")).show(truncate=False)

print("Relationship counts")
g.edges.groupBy("rel_type").count().orderBy(F.desc("count")).show(30, truncate=False)

## 7.1 Join degree results back to node attributes

GraphFrames returns degree DataFrames with only the node id and degree value.

To interpret the result, we join the degree data back to the vertices DataFrame.


In [ ]:
degrees = g.degrees
in_degrees = g.inDegrees
out_degrees = g.outDegrees

degree_summary = (
    g.vertices
    .join(degrees, on="id", how="left")
    .join(in_degrees, on="id", how="left")
    .join(out_degrees, on="id", how="left")
    .fillna({"degree": 0, "inDegree": 0, "outDegree": 0})
)

degree_summary.cache()

degree_summary.select(
    "id", "name", "node_type", "countries", "jurisdiction", "degree", "inDegree", "outDegree"
).orderBy(F.desc("degree")).show(20, truncate=80)

## 7.2 What do degree, in-degree, and out-degree mean here?

`degree` is the total number of edges connected to a node.

`inDegree` counts edges pointing **into** a node.  
`outDegree` counts edges going **out of** a node.

Because this graph is directed, the meaning depends on the node type:

| Node type | High in-degree means | High out-degree means |
|---|---|---|
| `address` | Many nodes point to the same address. This may indicate a reused registered address. | Usually not important here because addresses rarely point to other nodes. |
| `entity` | Many officers, intermediaries, or addresses point to the entity. This may indicate a heavily connected company/trust/foundation. | Usually low in this dataset because entities mostly receive links. |
| `intermediary` | Usually low here. | The intermediary points to many entities, so it acts as a service-provider hub. |
| `officer` | Usually low here. | The officer points to many entities, so it may be a repeated director, nominee, or corporate officer. |

So we should not interpret all high-degree nodes in the same way. A high-degree intermediary and a high-degree address are both hubs, but they are hubs for different reasons.


In [ ]:
degree_by_type = (
    degree_summary
    .groupBy("node_type")
    .agg(
        F.count("*").alias("num_nodes"),
        F.round(F.avg("degree"), 2).alias("avg_degree"),
        F.expr("percentile_approx(degree, 0.5)").alias("median_degree"),
        F.max("degree").alias("max_degree"),
        F.round(F.avg("inDegree"), 2).alias("avg_in_degree"),
        F.round(F.avg("outDegree"), 2).alias("avg_out_degree")
    )
    .orderBy(F.desc("avg_degree"))
)

degree_by_type.show(truncate=False)

## 7.3 Top nodes by node type

A single global ranking can be misleading because addresses, officers, intermediaries, and entities play different roles.

This table ranks the top nodes within each node type.


In [ ]:
w = Window.partitionBy("node_type").orderBy(F.desc("degree"))

top_by_type = (
    degree_summary
    .withColumn("rank_within_type", F.row_number().over(w))
    .filter(F.col("rank_within_type") <= 10)
    .select("rank_within_type", "node_type", "id", "name", "countries", "jurisdiction", "degree", "inDegree", "outDegree")
    .orderBy("node_type", "rank_within_type")
)

top_by_type.show(50, truncate=80)

## Interpreting the top nodes by type

This output shows four different kinds of hubs.

- The top **addresses** have high `inDegree` and zero `outDegree`. This means many records point to the same address.
- The top **entities** also have high `inDegree`. This means many officers, intermediaries, or address relationships point into those entities.
- The top **intermediaries** have high `outDegree`. This means they point to many entities and act as service-provider hubs.
- The top **officers** also have high `outDegree`. This means they are linked to many entities.

For example, in the displayed output, `MOSSACK FONSECA & CO. (PERU) CORP.` has 439 outgoing edges. In this graph, that means it is linked as an intermediary to 439 entities.

This is why direction matters: `inDegree` and `outDegree` tell us the role a node plays, not just how many links it has.


# 8. Degree distribution and long-tail networks

Many real-world graphs have a **long-tailed degree distribution**:

- Most nodes have very few connections.
- A small number of nodes have many connections.

This matters because averages can hide the structure of the network. The average degree may be low, while a few intermediaries, entities, officers, or addresses are still highly connected.


In [ ]:
degree_distribution = (
    degree_summary
    .groupBy("degree")
    .count()
    .orderBy("degree")
)

degree_distribution.show(30)

print("Highest degree values")
degree_distribution.orderBy(F.desc("degree")).show(20)

## 8.1 Plotting the degree distribution

For a quick visualization, we collect only the aggregated degree distribution to Pandas.

We do **not** collect the full raw graph.


In [ ]:
import matplotlib.pyplot as plt

degree_pd = degree_distribution.orderBy("degree").toPandas()

plt.figure(figsize=(8, 5))
plt.scatter(degree_pd["degree"], degree_pd["count"])
plt.yscale("log")
plt.xlabel("Degree")
plt.ylabel("Number of nodes, log scale")
plt.title("Degree distribution: many low-degree nodes, few high-degree hubs")
plt.show()


## Interpreting the long-tail plot

The y-axis uses a log scale because the counts are very uneven.

The points on the far left show that most nodes have very low degree. The isolated points far to the right show rare hubs with very high degree.

This shape tells us why ranking is useful: most nodes are not very connected, so we need degree, filters, and motifs to find the parts of the graph where structure is concentrated.


# 9. Graph density

A graph can have many edges and still be sparse.

For a directed graph with `n` vertices, the maximum possible number of directed edges is:

```text
n * (n - 1)
```

Density is:

```text
actual_edges / possible_edges
```

A low density means most possible relationships do not exist.


In [ ]:
num_vertices = g.vertices.count()
num_edges = g.edges.count()

density_directed = num_edges / (num_vertices * (num_vertices - 1)) if num_vertices > 1 else 0

print(f"Number of vertices: {num_vertices:,}")
print(f"Number of edges: {num_edges:,}")
print(f"Directed graph density: {density_directed:.12f}")

The density is extremely close to zero.

This means the graph is sparse: only a tiny fraction of all possible node-to-node links actually exist. That does **not** mean the graph is uninteresting. Sparse graphs can still have hubs, clusters, and repeated patterns.


# 10. Filtering the graph into meaningful subgraphs

The full graph is useful, but specific questions need smaller graph views.

Here we avoid ownership relationships because this CSV export does not contain useful ownership edges. Instead, we focus on the relationships that actually appear in the data:

- `intermediary_of`
- `registered_address`
- `officer_of`


## 10.1 Intermediary/entity graph

Question:

> Which intermediaries are linked to the most entities?

For this subgraph, a high `outDegree` intermediary means that the intermediary is connected to many entities.


In [ ]:
intermediary_g = (
    g
    .filterEdges("rel_type = 'intermediary_of'")
    .dropIsolatedVertices()
)

intermediary_degree = (
    intermediary_g.outDegrees
    .join(
        intermediary_g.vertices.select("id", "name", "countries", "node_type"),
        on="id",
        how="left"
    )
    .filter(F.col("node_type") == "intermediary")
    .orderBy(F.desc("outDegree"))
)

intermediary_degree.select(
    "id", "name", "countries", "outDegree"
).show(20, truncate=80)


This is the useful output for the intermediary graph.

It tells us which intermediaries are linked to the most entities. These are the main intermediary hubs in the filtered graph.


In [ ]:
intermediary_country_summary = (
    intermediary_degree
    .filter(F.col("countries").isNotNull())
    .withColumn("country", F.explode(F.split(F.col("countries"), ";")))
    .withColumn("country", F.trim(F.col("country")))
    .groupBy("country")
    .agg(
        F.count("*").alias("num_intermediaries"),
        F.sum("outDegree").alias("total_entities_linked"),
        F.max("outDegree").alias("max_intermediary_out_degree")
    )
    .orderBy(F.desc("total_entities_linked"))
)

intermediary_country_summary.show(20, truncate=False)


This country summary is more useful than just counting intermediaries.

`total_entities_linked` tells us how many intermediary-to-entity links are associated with intermediaries from each country. `max_intermediary_out_degree` shows whether one very large intermediary is driving the country total.


## 10.2 Address/entity graph

Question:

> Which addresses are reused the most?

For this subgraph, a high `inDegree` address means many nodes point to the same address.


In [ ]:
address_g = (
    g
    .filterEdges("rel_type = 'registered_address'")
    .dropIsolatedVertices()
)

address_hubs = (
    address_g.inDegrees
    .join(
        address_g.vertices.select("id", "name", "countries", "node_type"),
        on="id",
        how="left"
    )
    .filter(F.col("node_type") == "address")
    .orderBy(F.desc("inDegree"))
)

address_hubs.select(
    "id", "name", "countries", "inDegree"
).show(20, truncate=80)


This shows the most reused registered addresses.

A high `inDegree` address can indicate an administrative hub: many entities or officers are linked to the same address.


In [ ]:
address_country_summary = (
    address_hubs
    .filter(F.col("countries").isNotNull())
    .withColumn("country", F.explode(F.split(F.col("countries"), ";")))
    .withColumn("country", F.trim(F.col("country")))
    .groupBy("country")
    .agg(
        F.count("*").alias("num_addresses"),
        F.sum("inDegree").alias("total_registered_address_links"),
        F.max("inDegree").alias("max_address_in_degree")
    )
    .orderBy(F.desc("total_registered_address_links"))
)

address_country_summary.show(20, truncate=False)


This shows which countries contain the most reused address nodes.

Again, the `max_address_in_degree` column matters. A country total may be driven by one highly reused address.


## 10.3 Officer/entity graph

Question:

> Which officers are linked to the most entities?

For this subgraph, a high `outDegree` officer means the officer is connected to many entities.


In [ ]:
officer_g = (
    g
    .filterEdges("rel_type = 'officer_of'")
    .dropIsolatedVertices()
)

officer_degree = (
    officer_g.outDegrees
    .join(
        officer_g.vertices.select("id", "name", "countries", "node_type"),
        on="id",
        how="left"
    )
    .filter(F.col("node_type") == "officer")
    .orderBy(F.desc("outDegree"))
)

officer_degree.select(
    "id", "name", "countries", "outDegree"
).show(20, truncate=80)


This table identifies officers linked to many entities.

Some of these may be individuals; others may be corporate service companies or nominee structures. The graph result shows repeated connectivity, not legal responsibility.


## 10.4 Portugal-focused graph

Because this class is taught in Portugal, we also create a small Portugal-focused view.

First we keep only nodes whose own `countries` field contains Portugal.


In [ ]:
COUNTRY_OF_INTEREST = "Portugal"

portugal_only_g = (
    g
    .filterVertices(f"countries LIKE '%{COUNTRY_OF_INTEREST}%'")
    .dropIsolatedVertices()
)

print("Portugal-only graph")
print("Vertices:", portugal_only_g.vertices.count())
print("Edges:", portugal_only_g.edges.count())

portugal_only_g.vertices.groupBy("node_type").count().orderBy(F.desc("count")).show()
portugal_only_g.edges.groupBy("rel_type").count().orderBy(F.desc("count")).show(truncate=False)


This strict version keeps only Portugal-labelled nodes. It may be small because it removes neighbours that are connected to Portugal-labelled nodes but do not themselves have Portugal in the country field.

So we also create a broader Portugal neighbourhood.


In [ ]:
portugal_ids = (
    g.vertices
    .filter(F.col("countries").contains(COUNTRY_OF_INTEREST))
    .select(F.col("id").alias("portugal_id"))
    .distinct()
)

edges_from_portugal = g.edges.join(
    portugal_ids,
    g.edges.src == portugal_ids.portugal_id,
    "left_semi"
)

edges_to_portugal = g.edges.join(
    portugal_ids,
    g.edges.dst == portugal_ids.portugal_id,
    "left_semi"
)

portugal_neighbourhood_edges = (
    edges_from_portugal
    .unionByName(edges_to_portugal)
    .dropDuplicates(["src", "dst", "rel_type"])
)

portugal_neighbour_ids = (
    portugal_neighbourhood_edges.select(F.col("src").alias("id"))
    .union(portugal_neighbourhood_edges.select(F.col("dst").alias("id")))
    .distinct()
)

portugal_neighbourhood_vertices = g.vertices.join(
    portugal_neighbour_ids,
    on="id",
    how="inner"
)

portugal_neighbourhood_g = GraphFrame(
    portugal_neighbourhood_vertices,
    portugal_neighbourhood_edges
)

print("Portugal neighbourhood graph")
print("Vertices:", portugal_neighbourhood_g.vertices.count())
print("Edges:", portugal_neighbourhood_g.edges.count())

portugal_neighbourhood_g.vertices.groupBy("node_type").count().orderBy(F.desc("count")).show()
portugal_neighbourhood_g.edges.groupBy("rel_type").count().orderBy(F.desc("count")).show(truncate=False)


In [ ]:
portugal_degree_summary = (
    portugal_neighbourhood_g.degrees
    .join(
        portugal_neighbourhood_g.vertices.select(
            "id", "name", "node_type", "countries", "jurisdiction"
        ),
        on="id",
        how="left"
    )
    .orderBy(F.desc("degree"))
)

portugal_degree_summary.show(20, truncate=80)


The neighbourhood version is usually more useful.

It shows Portugal-labelled nodes together with the nodes directly connected to them. This gives a local view of the graph around Portugal.


# 11. Country-level graph summaries

The `countries` column is useful when combined with graph structure.

A country count alone is descriptive. A country combined with degree, node type, or relationship type is more informative.


In [ ]:
vertices_countries = (
    g.vertices
    .filter(F.col("countries").isNotNull())
    .withColumn("country", F.explode(F.split(F.col("countries"), ";")))
    .withColumn("country", F.trim(F.col("country")))
    .filter(F.col("country") != "")
)

country_node_counts = (
    vertices_countries
    .groupBy("country", "node_type")
    .count()
    .orderBy(F.desc("count"))
)

country_node_counts.show(30, truncate=False)


In [ ]:
degrees_with_country = (
    degree_summary
    .filter(F.col("countries").isNotNull())
    .withColumn("country", F.explode(F.split(F.col("countries"), ";")))
    .withColumn("country", F.trim(F.col("country")))
    .filter(F.col("country") != "")
)

country_degree_summary = (
    degrees_with_country
    .groupBy("country", "node_type")
    .agg(
        F.count("*").alias("num_nodes"),
        F.round(F.avg("degree"), 2).alias("avg_degree"),
        F.max("degree").alias("max_degree")
    )
    .filter(F.col("num_nodes") >= 10)
    .orderBy(F.desc("avg_degree"))
)

country_degree_summary.show(30, truncate=False)


This table is useful because it separates scale from connectivity.

A country may have many nodes but low average degree. Another country may have fewer nodes but one very connected hub.


# 12. Motif finding

Motifs are small graph patterns.

Instead of only ranking nodes, motif finding asks:

> Can we find this specific structure in the graph?

GraphFrames returns motif results as a DataFrame, so we can filter and summarise them with normal PySpark operations.


## Motif 1: one intermediary linked to two entities

Pattern:

`intermediary → entity A`  
`intermediary → entity B`

This finds examples of intermediaries that connect multiple entities.


In [ ]:
shared_intermediary = (
    g.find("(i)-[e1]->(a); (i)-[e2]->(b)")
    .filter("i.node_type = 'intermediary'")
    .filter("a.node_type = 'entity'")
    .filter("b.node_type = 'entity'")
    .filter("e1.rel_type = 'intermediary_of'")
    .filter("e2.rel_type = 'intermediary_of'")
    .filter("a.id < b.id")
)

shared_intermediary.select(
    F.col("i.name").alias("intermediary"),
    F.col("i.countries").alias("intermediary_countries"),
    F.col("a.name").alias("entity_a"),
    F.col("b.name").alias("entity_b")
).show(20, truncate=80)


This turns a high-degree intermediary into concrete examples: entity pairs connected through the same intermediary.


## Motif 2: two nodes sharing the same registered address

Pattern:

`node A → address`  
`node B → address`

This finds pairs of nodes linked to the same address.


In [ ]:
shared_address = (
    g.find("(a)-[e1]->(addr); (b)-[e2]->(addr)")
    .filter("addr.node_type = 'address'")
    .filter("e1.rel_type = 'registered_address'")
    .filter("e2.rel_type = 'registered_address'")
    .filter("a.id < b.id")
)

shared_address.select(
    F.col("addr.name").alias("shared_address"),
    F.col("addr.countries").alias("address_country"),
    F.col("a.name").alias("node_a"),
    F.col("a.node_type").alias("node_a_type"),
    F.col("b.name").alias("node_b"),
    F.col("b.node_type").alias("node_b_type")
).show(20, truncate=80)


This motif identifies shared-address structures.

A shared address is not proof of wrongdoing. It is a structural pattern that may indicate administrative reuse or a cluster worth inspecting.


# 13. Connected components

A connected component is a group of nodes connected by some path.

If two nodes are in the same connected component, it means there is a route between them through the graph. Running connected components on the full graph can take too long, so we demonstrate the concept with a tiny graph.


In [ ]:
component_vertices = spark.createDataFrame([
    ("a", "Node A"),
    ("b", "Node B"),
    ("c", "Node C"),
    ("d", "Node D"),
    ("e", "Node E"),
], ["id", "name"])

component_edges = spark.createDataFrame([
    ("a", "b", "connected_to"),
    ("b", "c", "connected_to"),
    ("d", "e", "connected_to"),
], ["src", "dst", "rel_type"])

component_g = GraphFrame(component_vertices, component_edges)

sc.setCheckpointDir("/tmp/graphframes-checkpoints")

component_results = component_g.connectedComponents()

component_results.orderBy("component", "id").show()


The tiny graph has two components:

- A, B, and C are connected to each other.
- D and E are connected to each other.
- There is no path between those two groups.

The same idea applies to the full graph, but the full computation is heavier.


In [ ]:
# Connected components on the full graph can take a long time.
# In our class we keep this code commented out.

# sc.setCheckpointDir("/tmp/graphframes-checkpoints")
# full_components = g.connectedComponents()

# component_sizes = (
#     full_components
#     .groupBy("component")
#     .count()
#     .orderBy(F.desc("count"))
# )

# component_sizes.show(20, truncate=False)


# 14. What we learned

In this class we built and explored the Panama Papers graph using GraphFrames.

The important points are:

1. Direction matters: `inDegree` and `outDegree` mean different things for different node types.
2. Addresses and entities mostly receive links in this graph.
3. Intermediaries and officers mostly send links to entities.
4. The graph has a long tail: most nodes have low degree, while a few nodes are major hubs.
5. `filterEdges` and `filterVertices` help us move from a large graph to focused questions.
6. Country summaries are useful when combined with node type and degree.
7. Motifs help us search for repeated structures.
8. Connected components are conceptually useful, but expensive on the full graph.

In the next class, we use PageRank, shortest paths, label propagation, SCC, and triangle count.
